# Step 19b — Post-hoc analysis of the Step 19 run (CPU only, ~15 min)

**Setup.** Mount the committed Step 19 output (**Add Data → Your Work → Notebooks**), which contains `step19_logits.npz`. No GPU is needed.

**Checks.**
1. Generated class frequencies against the classifier's own label frequencies on the test set, with binomial intervals: is the over-generation real?
2. Split-half reliability of $h$ for the reference and for the model at equal sample size.
3. The model's $h$ after rebalancing its trajectory weights so the two classes of each pair carry equal mass. Does the model's fold geometry match the data's once its class-frequency bias is removed?
4. The corrected figures.

In [ ]:
import glob, numpy as np, matplotlib.pyplot as plt
from itertools import combinations
from scipy.signal import fftconvolve
from scipy.stats import spearmanr
plt.rcParams['figure.dpi'] = 120; rng = np.random.default_rng(7)
path = glob.glob('/kaggle/input/**/step19_logits.npz', recursive=True); assert path, 'mount the Step 19 output'
Z = np.load(path[0]); L_TE, W_TE, L_TR, W_TR, L_M, W_M, SIG_REC = [Z[k] for k in ['L_TE', 'W_TE', 'L_TR', 'W_TR', 'L_M', 'W_M', 'SIG_REC']]
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']; VEH = {0, 1, 8, 9}; PAIRS = list(combinations(range(10), 2))
G = np.linspace(-4.0, 4.0, 4001); DGG = G[1]-G[0]
def wkde_score(m, w):
    m = np.clip(m, -8, 8); w = np.asarray(w, dtype=np.float64); neff = w.sum()**2/np.maximum((w**2).sum(), 1e-12)
    mu = np.average(m, weights=w); sd = np.sqrt(np.average((m-mu)**2, weights=w)); bw = max(1.5*1.06*sd*neff**(-1/5), 1e-3)
    h, _ = np.histogram(m, bins=np.append(G-DGG/2, G[-1]+DGG/2), weights=w, density=True); k = np.exp(-G**2/(2*bw**2)); k /= k.sum()
    p = np.maximum(fftconvolve(h, k, 'same'), 1e-300); return np.gradient(np.log(p), G), p
def _merge(pts, merge=0.03):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary(sm, p, lo=-3.5, hi=3.5, prominence=0.05):
    keep = (G > lo) & (G < hi) & (p > 1e-3*p.max()); idx = np.where((np.sign(sm[:-1]) != np.sign(sm[1:])) & keep[:-1] & keep[1:])[0]
    pts = _merge([(float(G[i]-sm[i]*DGG/(sm[i+1]-sm[i])) if sm[i+1] != sm[i] else float(G[i]), 'max' if sm[i] > 0 else 'min') for i in idx])
    logp = np.concatenate([[0.0], np.cumsum(0.5*(sm[1:]+sm[:-1])*DGG)]); lp = lambda x: float(np.interp(x, G, logp)); ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if pts[a][1] != pts[a+1][1] and abs(lp(pts[a][0]) - lp(pts[a+1][0])) < prominence: del pts[a:a+2]; ch = True; break
    return pts
def n_max(pts): return sum(k == 'max' for _, k in pts)
def cusp_fit(sm, p, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(G) < window) & (p > 0); c3, c2, c1, c0 = np.polyfit(G[ok], sm[ok], 3, w=np.sqrt(p[ok])); b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    return dict(h=c0 + c1*s + c2*s**2 + c3*s**3, b=b)
def scan_transition(dist_at):
    prev = None
    for i, s in enumerate(SIG_REC):
        d = dist_at(i); pts = [(0.0, 'max')] if d is None else stationary(*d)
        if n_max(pts) >= 2 and prev is not None: return float(s), i
        if n_max(pts) == 1: prev = 1
    return None
def std_stats(z, wa, wb):
    mua, mub = np.average(z, weights=wa), np.average(z, weights=wb); half = 0.5*(mua-mub); sda = np.sqrt(np.average((z-mua)**2, weights=wa)); sdb = np.sqrt(np.average((z-mub)**2, weights=wb))
    return 0.5*(mua+mub), half, abs(half)/(0.5*(sda+sdb))
def pair_std(L, W, a, b): return [std_stats(L[g, :, a].astype(np.float64) - L[g, :, b], W[:, a] + 1e-12, W[:, b] + 1e-12) for g in range(len(SIG_REC))]
def pair_h(L, W, a, b, STD_, idx, balance=False):
    Wa = W[idx, a].astype(np.float64); Wb = W[idx, b].astype(np.float64)
    if balance: Wa = Wa/Wa.sum()*0.5*(Wa.sum()+Wb.sum()); Wb = Wb/Wb.sum()*0.5*(Wa.sum()+Wb.sum())
    def dist(g):
        mid, half, sep = STD_[g]
        if sep < 0.3: return None
        z = (L[g, idx, a].astype(np.float64) - L[g, idx, b]); return wkde_score((z - mid)/half, Wa + Wb)
    tr = scan_transition(dist)
    if tr is None: return None, None
    sm, p = dist(tr[1]); return tr[0], cusp_fit(sm, p)['h']
print('loaded', {k: Z[k].shape for k in Z.files})

## 1. Is the model's class imbalance real? (generated vs classifier-labelled test frequencies)

In [ ]:
f_gen = np.bincount(W_M.argmax(1), minlength=10)/len(W_M); f_tst = np.bincount(W_TE.argmax(1), minlength=10)/len(W_TE); n = len(W_M)
se = np.sqrt(f_gen*(1-f_gen)/n + f_tst*(1-f_tst)/len(W_TE)); z = (f_gen - f_tst)/se
print(f"{'class':>11} {'generated':>10} {'classifier on test':>19} {'ratio':>6} {'z':>6}")
for c in np.argsort(-(f_gen/f_tst)): print(f"{CLASSES[c]:>11} {f_gen[c]:10.3f} {f_tst[c]:19.3f} {f_gen[c]/f_tst[c]:6.2f} {z[c]:+6.1f}")
var = np.array([0.2431, 0.2756, 0.2109, 0.2605, 0.1779, 0.2418, 0.1968, 0.2363, 0.2152, 0.2516])
print(f"corr(log over-generation ratio, class pixel variance): pearson {np.corrcoef(np.log(f_gen/f_tst), var)[0,1]:.2f}, spearman {spearmanr(f_gen/f_tst, var).correlation:.2f} (10 classes)")
OVER = np.log(f_gen/f_tst)

## 2–3. Split-half reliability of h, and the model's h after rebalancing class mass

In [ ]:
NM = len(W_M); half_n = NM//2; rows = []
for a, b in PAIRS:
    STD_ = pair_std(L_TE, W_TE, a, b)
    im = rng.permutation(NM); ir = rng.permutation(len(W_TE))[:NM]
    s1, hm1 = pair_h(L_M, W_M, a, b, STD_, im[:half_n]); s2, hm2 = pair_h(L_M, W_M, a, b, STD_, im[half_n:])
    s3, hr1 = pair_h(L_TE, W_TE, a, b, STD_, ir[:half_n]); s4, hr2 = pair_h(L_TE, W_TE, a, b, STD_, ir[half_n:])
    _, hm = pair_h(L_M, W_M, a, b, STD_, np.arange(NM)); _, hr = pair_h(L_TE, W_TE, a, b, STD_, ir)
    _, hmb = pair_h(L_M, W_M, a, b, STD_, np.arange(NM), balance=True); _, hrb = pair_h(L_TE, W_TE, a, b, STD_, ir, balance=True)
    rows.append(dict(pair=(a, b), hm1=hm1, hm2=hm2, hr1=hr1, hr2=hr2, hm=hm, hr=hr, hmb=hmb, hrb=hrb))
def corr(x, y):
    x = np.array(x, float); y = np.array(y, float); ok = np.isfinite(x) & np.isfinite(y); return np.corrcoef(x[ok], y[ok])[0, 1], ok.sum()
print("split-half reliability (corr between halves, n = %d each):  model h %.2f (n=%d);  reference h %.2f (n=%d)" % ((half_n,) + corr([r['hm1'] for r in rows], [r['hm2'] for r in rows]) + corr([r['hr1'] for r in rows], [r['hr2'] for r in rows])))
print("model vs reference h:            raw %.2f (n=%d);  both rebalanced %.2f (n=%d)" % (corr([r['hm'] for r in rows], [r['hr'] for r in rows]) + corr([r['hmb'] for r in rows], [r['hrb'] for r in rows])))
dover = np.array([OVER[a]-OVER[b] for a, b in PAIRS]); dvar = np.array([var[a]-var[b] for a, b in PAIRS])
print("model h vs log over-generation difference: raw %.2f; rebalanced %.2f" % (corr([r['hm'] for r in rows], dover)[0], corr([r['hmb'] for r in rows], dover)[0]))
print("h vs class-variance difference:  reference %.2f, model raw %.2f, model rebalanced %.2f" % (corr([r['hr'] for r in rows], dvar)[0], corr([r['hm'] for r in rows], dvar)[0], corr([r['hmb'] for r in rows], dvar)[0]))
lv = lambda key: sum(1 for r, dv in zip(rows, dvar) if r[key] is not None and np.sign(r[key]) == -np.sign(dv))
print("lower-variance class persists:  reference %d/44;  model raw %d/44;  model rebalanced %d/44" % (lv('hr'), lv('hm'), lv('hmb')))
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
ax[0].scatter([r['hm1'] for r in rows], [r['hm2'] for r in rows], s=14, label='model'); ax[0].scatter([r['hr1'] for r in rows], [r['hr2'] for r in rows], s=14, label='reference'); ax[0].axline((0, 0), slope=1, color='k', lw=0.6); ax[0].set_title('split-half reliability of h'); ax[0].legend(fontsize=7)
ax[1].scatter([r['hr'] for r in rows], [r['hm'] for r in rows], s=14, label='raw'); ax[1].scatter([r['hrb'] for r in rows], [r['hmb'] for r in rows], s=14, label='class mass rebalanced'); ax[1].axline((0, 0), slope=1, color='k', lw=0.6); ax[1].set_xlabel('h reference'); ax[1].set_ylabel('h model'); ax[1].legend(fontsize=7); ax[1].set_title('model vs data asymmetry')
ax[2].scatter(dover, [r['hm'] for r in rows], s=14); ax[2].axhline(0, color='k', lw=0.5); ax[2].axvline(0, color='k', lw=0.5); ax[2].set_xlabel('log over-generation(a) - log over-generation(b)'); ax[2].set_ylabel('h model (raw)'); ax[2].set_title('model asymmetry vs its class bias')
plt.tight_layout(); plt.show()

## 4. Corrected transition figure (bootstrap-median point estimates, errors clipped at zero)

In [ ]:
sr, sm_, lo_, hi_, kind = [], [], [], [], []
for a, b in PAIRS:
    STD_ = pair_std(L_TE, W_TE, a, b); bm, br = [], []
    for _ in range(30):
        s_m, _ = pair_h(L_M, W_M, a, b, STD_, rng.integers(NM, size=NM)); s_r, _ = pair_h(L_TE, W_TE, a, b, STD_, rng.choice(len(W_TE), size=NM, replace=True))
        if s_m and s_r: bm.append(s_m); br.append(s_r)
    if len(bm) < 10: continue
    rat = np.array(bm)/np.array(br); sr.append(np.median(br)); sm_.append(np.median(bm)); lo_.append(np.percentile(rat, 2.5)); hi_.append(np.percentile(rat, 97.5))
    kind.append('va' if ((a in VEH) != (b in VEH)) else 'vv' if a in VEH else 'aa')
sr, sm_, lo_, hi_ = map(np.array, (sr, sm_, lo_, hi_)); R = sm_/sr
print(f"{len(sr)} pairs: median ratio {np.median(R):.2f}; CI above 1: {(lo_ > 1).sum()}, below 1: {(hi_ < 1).sum()}, overlapping: {((lo_ <= 1) & (hi_ >= 1)).sum()}")
col = ['C0' if k == 'va' else 'C1' if k == 'vv' else 'C2' for k in kind]; fig, ax = plt.subplots(figsize=(4.8, 4.2))
ax.errorbar(sr, sm_, yerr=[np.clip(sm_ - lo_*sr, 0, None), np.clip(hi_*sr - sm_, 0, None)], fmt='none', ecolor='grey', lw=0.6); ax.scatter(sr, sm_, s=18, c=col)
lim = [min(sr.min(), sm_.min())*0.8, max(sr.max(), sm_.max())*1.2]; ax.plot(lim, lim, 'k--', lw=0.7); ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('sigma_bif, reference (test set)'); ax.set_ylabel('sigma_bif, pretrained EDM (reverse SDE)'); ax.set_title('blue vehicle/animal, orange vehicle/vehicle, green animal/animal', fontsize=8)
plt.tight_layout(); plt.show()

## What to report back
- The frequency table and the variance correlation.
- The reliability line, the two model-vs-reference $h$ lines, the over-generation lines, the variance lines and the lower-variance counts.
- The corrected transition summary line and the figures.

**The decisive number** is model-vs-reference $h$ after rebalancing class mass:
- **If it rises to near the reference's reliability**, the model's fold geometry matches the data. Its asymmetry is then entirely its own class-frequency bias, which gives a clean mechanistic link between the fold and generation bias in a production model.
- **If it stays near zero**, the fold geometry differs too.